---

# Intelligent Agents and Search
# Programming Assignment 2

<br> 
In this assignment, you will implement a genetic algorithm to solve a simple problem - matching a target string for a recipe. The assignment is meant to illustrate the workings of the algorithm, and provide you with an opportunity to play with the algorithm's parameters and notice how they impact the convergence and final solution quality.

### Your name:

<br> 

---

Some useful packages and libraries:



In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from collections import deque
import heapq
import unittest
from scipy import stats
import copy as cp
from time import time

---

## Problem 1: Maximizing an Objective Function with a Genetic Algorithm 

In this problem, you're going to use a genetic algorithm to re-create a target solution string. Unlike some problems where the final solution is unknown, here, the solution is known and the assignment objective is to become more familiar with the GA mechanisms. 

Suppose we've lost the index card with our favorite cupcake recipe. We know the ingredients of the cake, but cannot remember the exact amount of each ingredient. We decide to use a GA to generate the ingredient amounts. With each iteration of the GA, we bake the cupcakes and taste-test them. We achieve our goal and stop running the GA when we get to the actual recipe: 

* 1 tsp salt 
* 3 tsp baking powder 
* 2 cups all-purpose flour 
* 1 cup butter 
* 1 cup granulated sugar 
* 4 large eggs
* 1 tsp vanilla extract
* 1 cup buttermilk 

In [34]:
target = [1, 3, 2, 1, 1, 4, 1, 1]

An example starting state for a member of our population might look like: $state = [1, 2, 100, 36, 60, 3, 5, 50]$, which is equivalent to 1 tsp salt, 2 tsp baking powder, 100 cups of flour, 36 cups of butter, 60 cups of granulated sugar, 3 large eggs, 5 tsp vanilla, and 50 cups of buttermilk.

### (1a) 

Write an objective function `def recipe_success(state)` that takes a single argument state, and returns the objective function value (fitness) of the state. **The objective function should be maximized when a state reaches the target.** You could for example define the fitness score of a particular state based on how far away each entry is from the target recipe.

In [35]:
def recipe_success(state):
    """
    Fitness/objective function to maximize.

    We want to *match the target recipe exactly*. We score a candidate state by
    measuring its total absolute error from the target vector.

    - Perfect match => error = 0 => score = 0  (best possible)
    - Worse matches => more error => score is more negative

    Returning a value we *maximize*:
        score(state) = - sum_i |state[i] - target[i]|
    """
    # Defensive: allow lists or numpy arrays
    
    # your code here
    total_abs_error = 0
    for i in range(len(state)):
        total_abs_error += abs(state[i] - target[i])

    return -total_abs_error


In [36]:
# Hidden Tests

### (1b) 

Write a GA that starts with a population of 100 randomly generated "recipes/states/members" and uses the objective function you wrote in **(1a)** to hopefully hit the target after a certain number of generations. 

The skeleton code for your GA is provided for you. 

Key components of your code:
- Generate the initial population randomly from integers between 0 and 100 
- Allow for mutations in your population with an overall probability of mutation set to p = 0.2
- Choose 2 "parents" in the generation of each "child"
- Choose a random split point at which to combine the two "parents"
- Run the algorithm for 50 iterations ("generations"). Do you hit your target?

In [ ]:
class problem:
    def __init__(self, initial_population, objective_function, mutation_probability, fitness_goal,
                 gene_min=1, gene_max=4, rng=None):
        '''
        initial_population = list of lists; each sub-list is a dna string for a population member
        objective_function = objective function to maximize
        mutation_probability = probability that any given child has a mutation
        fitness_goal = fitness goal to achieve (stopping criterion, once member reaches this)
        gene_min/gene_max = inclusive bounds for genes (integers)
        rng = optional numpy random Generator for reproducibility
        '''
        # your code here
        self.population = initial_population
        self.objective_function = objective_function
        self.mutation_probability = mutation_probability
        self.fitness_goal = fitness_goal
        self.gene_min = gene_min
        self.gene_max = gene_max
        self.rng = rng if rng is not None else np.random.default_rng()
        
    def fitness(self):
        '''
        Calculate each population member's probability of being selected for
        reproduction based on performance on objective function.
        Returns an array p_reproduce of length |population| that sums to 1.
        '''
        # your code here
        population_fitness = []
        for population in self.population:
            population_fitness.append(self.objective_function(population))

        maximum_fitness = max(population_fitness)
        minimum_fitness = min(population_fitness)

        buffer = (maximum_fitness - minimum_fitness) * 0.05 \
            if maximum_fitness != minimum_fitness \
            else 1e-6
        adjusted_fitness = [
            (x - minimum_fitness) + buffer
            for x in population_fitness
        ]

        total_sum = sum(adjusted_fitness)
        p_reproduce = [x / total_sum for x in adjusted_fitness]
        
        return p_reproduce

    def reproduce(self, parent1, parent2):
        '''
        One-point crossover:
        - choose a split point k in [1, dna_len-1]
        - child = parent1[:k] + parent2[k:]
        '''
        # your code here
        split_point = self.rng.integers(low=0, high=len(parent1))
        child = parent1[:split_point] + parent2[split_point:]

        return child

    def mutate(self, child):
        '''
        With probability mutation_probability, randomly choose ONE gene and resample it.
        '''
        # your code here
        should_mutate = self.rng.uniform(0, 1) > self.mutation_probability
        if should_mutate:
            resample_index = self.rng.integers(0, len(child))
            child[resample_index] = rng.integers(self.gene_min, self.gene_max)
        
        return child

In [38]:
def genetic_algorithm(genetic_problem, num_generations, verbose=False):
    """Run a simple genetic algorithm.

    Returns a dictionary with:
      - 'best_member'
      - 'best_score'
      - 'generation_found' (None if not reached)
      - 'history_best' (best score per generation)
      - 'history_avg' (avg score per generation)
      - 'num_evaluated' (objective function evaluations)
    """
    pop_size = len(genetic_problem.population)
    rng = genetic_problem.rng

    best_member = None
    best_score = -float("inf")
    generation_found = None
    history_best = []
    history_avg = []
    num_evaluated = 0

    for gen in range(int(num_generations)):
        # Evaluate population
        scores = [genetic_problem.objective_function(m) for m in genetic_problem.population]
        num_evaluated += len(scores)

        gen_best_idx = int(np.argmax(scores))
        gen_best_score = scores[gen_best_idx]
        gen_best_member = genetic_problem.population[gen_best_idx]

        if gen_best_score > best_score:
            best_score = gen_best_score
            best_member = list(gen_best_member)

        history_best.append(best_score)
        history_avg.append(float(np.mean(scores)))

        if verbose:
            print(f"Gen {gen:03d}: best_score={best_score}, best_member={best_member}")

        # Check stopping condition
        if best_score >= genetic_problem.fitness_goal:
            generation_found = gen
            break

        # Selection probabilities
        p = genetic_problem.fitness()

        # Create next generation
        new_population = []
        for _ in range(pop_size):
            # pick two parents (with replacement) proportional to fitness
            i1, i2 = rng.choice(pop_size, size=2, replace=True, p=p)
            parent1 = genetic_problem.population[int(i1)]
            parent2 = genetic_problem.population[int(i2)]

            child = genetic_problem.reproduce(parent1, parent2)
            child = genetic_problem.mutate(child)
            new_population.append(child)

        genetic_problem.population = new_population

    return {
        "best_member": best_member,
        "best_score": best_score,
        "generation_found": generation_found,
        "history_best": history_best,
        "history_avg": history_avg,
        "num_evaluated": num_evaluated,
    }


In [44]:
# (1b) Run the GA on the recipe problem

# Reproducibility (optional but recommended for debugging)
seed = 42
rng = np.random.default_rng(seed)

# Population of 100 random recipes (each gene is an integer amount)
population_size = 100
dna_len = len(target)

# We keep the search space small (1..4) to match the target's scale.
gene_min, gene_max = 1, 4
initial_population = [
    [int(rng.integers(gene_min, gene_max + 1)) for _ in range(dna_len)]
    for _ in range(population_size)
]

genetic_problem = problem(
    initial_population=initial_population,
    objective_function=recipe_success,
    mutation_probability=0.2,
    fitness_goal=0,   # score 0 means exact match
    gene_min=gene_min,
    gene_max=gene_max,
    rng=rng,
)

out = genetic_algorithm(genetic_problem, 200, verbose=False)
print(out)


{'best_member': [1, 3, 2, 1, 1, 4, 1, 1], 'best_score': 0, 'generation_found': 7, 'history_best': [-3, -3, -2, -1, -1, -1, -1, 0], 'history_avg': [-11.33, -9.76, -8.18, -7.06, -6.0, -5.35, -5.31, -5.0], 'num_evaluated': 800}


In [40]:
# Hidden Tests